# NL-SQL Agent Development Notebook

This notebook is for developing and improving the NL-SQL agents:
- SQL Generator
- Ambiguity Checker
- Visualization Recommender

Use this notebook to:
- Test and iterate on prompts
- Experiment with different LLM parameters
- Debug agent behavior
- Profile agent performance
- Test edge cases

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# Core imports
from app.agents.nl_to_sql_agent import NLToSQLAgent
from app.agents.sql_generator import SQLGenerator
from app.agents.ambiguity_checker import AmbiguityChecker
from app.agents.visualization_recommender import VisualizationRecommender
from app.agents.prompts import PromptBuilder
from app.database import db_service
from app.config import settings

from openai import OpenAI
import json
import time
from typing import Dict, Any, List
import pandas as pd

print("✅ Imports successful")

## 2. Initialize Agents

Load the agent components for testing

In [ ]:
# Initialize main agent
agent = NLToSQLAgent()

# Get individual components for testing
sql_generator = agent.sql_generator
ambiguity_checker = agent.ambiguity_checker
viz_recommender = agent.visualization_recommender

# Load context for inspection
view_schemas = db_service.get_view_schemas()
semantic_dict = db_service.get_semantic_dictionary()
query_templates = db_service.get_query_templates()

print(f"✅ Agent initialized")
print(f"   - {len(view_schemas)} database views loaded")
print(f"   - {len(semantic_dict)} semantic terms loaded")
print(f"   - {len(query_templates)} query templates loaded")
print(f"   - Using model: {settings.OPENAI_MODEL}")

## 3. SQL Generator Testing

Test and develop the SQL generation logic

### 3.1 Inspect SQL Generation Prompt

In [ ]:
# View the current SQL generation prompt
sql_prompt = PromptBuilder.build_sql_generation_prompt(view_schemas, query_templates)
print("=" * 80)
print("SQL GENERATION SYSTEM PROMPT")
print("=" * 80)
print(sql_prompt)
print("=" * 80)
print(f"Prompt length: {len(sql_prompt)} characters")
print(f"Estimated tokens: ~{len(sql_prompt) // 4}")

### 3.2 Test SQL Generation

In [ ]:
def test_sql_generation(question: str, model: str = None, show_prompt: bool = False):
    """
    Test SQL generation with detailed output
    """
    print(f"\n{'='*80}")
    print(f"TESTING SQL GENERATION")
    print(f"{'='*80}")
    print(f"Question: {question}")
    print(f"Model: {model or settings.OPENAI_MODEL}")
    print(f"{'='*80}\n")
    
    start = time.time()
    result = sql_generator.generate_sql(question, model=model)
    elapsed = time.time() - start
    
    print(f"⏱️  Time: {elapsed:.2f}s")
    print(f"✅ Success: {result.get('success')}")
    
    if result.get('success'):
        print(f"🔢 Tokens: {result.get('tokens_used')}")
        print(f"\n📝 Generated SQL:\n{'-'*80}")
        print(result['sql'])
        print(f"{'-'*80}\n")
        
        # Validate SQL
        is_valid, error = db_service.validate_query(result['sql'])
        if is_valid:
            print("✅ SQL validation passed")
        else:
            print(f"❌ SQL validation failed: {error}")
    else:
        print(f"\n❌ Error: {result.get('error')}")
        if 'sql' in result:
            print(f"\n📝 Generated SQL (invalid):\n{'-'*80}")
            print(result['sql'])
            print(f"{'-'*80}\n")
    
    return result

# Test with a sample question
result = test_sql_generation("How many patients do we have?")

### 3.3 Test Complex Queries

In [ ]:
# Test increasingly complex queries
complex_queries = [
    "What is the average cost per patient?",
    "Show me the top 5 most expensive conditions",
    "How many readmissions within 30 days by state?",
    "What is the mortality rate for patients with diabetes?",
    "Show me utilization trends by age group and gender"
]

for query in complex_queries:
    result = test_sql_generation(query)
    print("\n" + "="*80 + "\n")
    time.sleep(1)  # Rate limiting

### 3.4 Compare Different Models

In [ ]:
def compare_models(question: str, models: List[str]):
    """
    Compare SQL generation across different models
    """
    print(f"\n{'='*80}")
    print(f"COMPARING MODELS FOR: {question}")
    print(f"{'='*80}\n")
    
    results = []
    for model in models:
        print(f"\n🤖 Testing {model}...")
        start = time.time()
        result = sql_generator.generate_sql(question, model=model)
        elapsed = time.time() - start
        
        results.append({
            'model': model,
            'success': result.get('success'),
            'time': elapsed,
            'tokens': result.get('tokens_used', 0),
            'sql': result.get('sql', '')[:100] + '...' if result.get('sql') else None
        })
        time.sleep(1)
    
    # Display comparison table
    df = pd.DataFrame(results)
    print("\n📊 COMPARISON RESULTS:\n")
    print(df.to_string(index=False))
    
    return results

# Example: Compare GPT-4o vs GPT-3.5
# compare_models(
#     "What are the top 5 conditions by patient count?",
#     models=["gpt-4o", "gpt-3.5-turbo"]
# )

## 4. Ambiguity Checker Testing

Test ambiguity detection logic

### 4.1 Inspect Ambiguity Check Prompt

In [ ]:
# View the ambiguity check prompt
ambiguity_prompt = PromptBuilder.build_ambiguity_check_prompt()
print("=" * 80)
print("AMBIGUITY CHECK SYSTEM PROMPT")
print("=" * 80)
print(ambiguity_prompt)
print("=" * 80)
print(f"Prompt length: {len(ambiguity_prompt)} characters")

### 4.2 Test Ambiguity Detection

In [ ]:
def test_ambiguity_check(query: str):
    """
    Test ambiguity detection
    """
    print(f"\n{'='*80}")
    print(f"TESTING AMBIGUITY CHECK")
    print(f"{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}\n")
    
    start = time.time()
    result = ambiguity_checker.check_for_ambiguity(query)
    elapsed = time.time() - start
    
    print(f"⏱️  Time: {elapsed:.2f}s")
    print(f"\n📋 Result:")
    print(json.dumps(result, indent=2))
    
    if result.get('is_ambiguous'):
        print(f"\n⚠️  Query is AMBIGUOUS")
        print(f"Reason: {result.get('reason')}")
        if result.get('clarification_options'):
            print(f"\n💡 Clarification options:")
            for i, option in enumerate(result['clarification_options'], 1):
                print(f"  {i}. {option.get('text')}")
                print(f"     → {option.get('refined_query')}")
    else:
        print(f"\n✅ Query is CLEAR")
    
    return result

# Test with clear query
test_ambiguity_check("How many patients do we have?")

# Test with ambiguous query
test_ambiguity_check("Show me the most expensive ones")

### 4.3 Test Edge Cases

In [ ]:
# Test ambiguous queries
ambiguous_test_cases = [
    "Show me the top patients",  # Ambiguous: top by what?
    "What about last year?",  # Ambiguous: what metric?
    "Compare them",  # Ambiguous: compare who/what?
    "Show me the average",  # Ambiguous: average of what?
    "How many patients do we have in California?",  # Clear
]

for query in ambiguous_test_cases:
    result = test_ambiguity_check(query)
    print("\n" + "="*80 + "\n")
    time.sleep(1)

## 5. Visualization Recommender Testing

Test visualization recommendation logic

### 5.1 Test Visualization Recommendation

In [ ]:
def test_visualization(sql: str, sample_rows: int = 5):
    """
    Test visualization recommendation for a SQL query
    """
    print(f"\n{'='*80}")
    print(f"TESTING VISUALIZATION RECOMMENDATION")
    print(f"{'='*80}")
    print(f"SQL:\n{sql}")
    print(f"{'='*80}\n")
    
    # Execute query
    result = db_service.execute_query(sql)
    
    if not result.get('success'):
        print(f"❌ Query execution failed: {result.get('error')}")
        return None
    
    rows = result.get('rows', [])
    columns = result.get('columns', [])
    
    print(f"✅ Query returned {len(rows)} rows, {len(columns)} columns")
    print(f"\n📊 Sample data:")
    if rows:
        df = pd.DataFrame(rows[:sample_rows])
        print(df.to_string(index=False))
    
    # Get visualization recommendation
    print(f"\n🎨 Recommending visualization...")
    start = time.time()
    viz = viz_recommender.recommend_visualization(
        sql=sql,
        columns=columns,
        results=rows,
        row_count=len(rows)
    )
    elapsed = time.time() - start
    
    print(f"⏱️  Time: {elapsed:.2f}s")
    print(f"\n📋 Recommendation:")
    print(json.dumps(viz, indent=2))
    
    # Get chart library config
    if viz.get('recommended_chart') and viz.get('chart_config'):
        chart_config = viz_recommender.get_chart_library_config(
            viz['recommended_chart'],
            viz['chart_config'],
            rows
        )
        print(f"\n📊 Chart.js Config:")
        print(json.dumps(chart_config, indent=2, default=str))
    
    return viz

# Test with a metric query
test_visualization("SELECT COUNT(*) as patient_count FROM vw_patients_2025")

# Test with a bar chart query
test_visualization("""
SELECT gender, COUNT(*) as count 
FROM vw_patients_2025 
GROUP BY gender 
ORDER BY count DESC
""")

### 5.2 Test Number Formatting

In [ ]:
def test_number_format(question: str, columns: List[str], sample_data: List[Dict]):
    """
    Test number format detection
    """
    print(f"\n{'='*80}")
    print(f"TESTING NUMBER FORMAT DETECTION")
    print(f"{'='*80}")
    print(f"Question: {question}")
    print(f"Columns: {columns}")
    print(f"Sample data: {sample_data[:3]}")
    print(f"{'='*80}\n")
    
    format_result = viz_recommender.detect_number_format(
        question=question,
        columns=columns,
        results=sample_data
    )
    
    print(f"📋 Detected format: {format_result}")
    return format_result

# Test currency detection
test_number_format(
    "What is the average cost per patient?",
    ["avg_cost"],
    [{"avg_cost": 15234.56}]
)

# Test percentage detection
test_number_format(
    "What is the readmission rate?",
    ["readmission_rate"],
    [{"readmission_rate": 0.23}]
)

## 6. Full Pipeline Testing

Test the complete agent pipeline

In [ ]:
def test_full_pipeline(question: str, skip_ambiguity: bool = False):
    """
    Test the complete NL-SQL pipeline
    """
    print(f"\n{'='*80}")
    print(f"TESTING FULL PIPELINE")
    print(f"{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    start = time.time()
    result = agent.execute_query(
        question,
        include_explanation=True,
        include_visualization=True,
        skip_ambiguity_check=skip_ambiguity
    )
    elapsed = time.time() - start
    
    print(f"⏱️  Total time: {elapsed:.2f}s\n")
    
    # Check if ambiguous
    if result.get('needs_clarification'):
        print(f"⚠️  Query needs clarification")
        print(f"Reason: {result.get('ambiguity', {}).get('reason')}")
        return result
    
    # Show SQL
    if result.get('sql'):
        print(f"📝 Generated SQL:\n{'-'*80}")
        print(result['sql'])
        print(f"{'-'*80}\n")
    
    # Show explanation
    if result.get('explanation'):
        print(f"💬 Explanation: {result['explanation']}\n")
    
    # Show results
    if result.get('success'):
        print(f"✅ Query executed successfully")
        print(f"   - {result.get('row_count', 0)} rows returned")
        print(f"   - {result.get('tokens_used', 0)} tokens used\n")
        
        # Show sample data
        rows = result.get('rows', [])
        if rows:
            print(f"📊 Sample results:")
            df = pd.DataFrame(rows[:5])
            print(df.to_string(index=False))
            print()
        
        # Show visualization
        if result.get('visualization'):
            viz = result['visualization']
            print(f"🎨 Visualization: {viz.get('recommended_chart')}")
            print(f"   Reason: {viz.get('reason')}")
            print(f"   Number format: {viz.get('number_format')}")
    else:
        print(f"❌ Query failed: {result.get('error')}")
    
    return result

# Test the full pipeline
test_full_pipeline("How many patients do we have?")

## 7. Prompt Engineering Experiments

Experiment with different prompt variations

In [ ]:
def test_custom_prompt(question: str, custom_system_prompt: str):
    """
    Test with a custom system prompt
    """
    client = OpenAI(
        api_key=settings.OPENAI_API_KEY,
        base_url=settings.OPENAI_API_BASE_URL
    )
    
    messages = [
        {"role": "system", "content": custom_system_prompt},
        {"role": "user", "content": question}
    ]
    
    response = client.chat.completions.create(
        model=settings.OPENAI_MODEL,
        messages=messages,
        temperature=0.0
    )
    
    sql = response.choices[0].message.content.strip()
    
    # Clean SQL
    if sql.startswith("```sql"):
        sql = sql.replace("```sql", "").replace("```", "").strip()
    
    print(f"\n{'='*80}")
    print(f"CUSTOM PROMPT TEST")
    print(f"{'='*80}")
    print(f"Question: {question}")
    print(f"\nGenerated SQL:\n{'-'*80}")
    print(sql)
    print(f"{'-'*80}\n")
    
    # Validate
    is_valid, error = db_service.validate_query(sql)
    if is_valid:
        print("✅ SQL is valid")
    else:
        print(f"❌ SQL validation failed: {error}")
    
    return sql

# Example: Test with a minimal prompt
# minimal_prompt = """
# You are a SQL expert. Convert natural language to PostgreSQL queries.
# Use the vw_patients_2025 view for patient data.
# Return only the SQL query, nothing else.
# """
# test_custom_prompt("How many patients?", minimal_prompt)

## 8. Performance Profiling

Profile agent performance and identify bottlenecks

In [ ]:
def profile_agent_pipeline(question: str, iterations: int = 3):
    """
    Profile the agent pipeline to identify bottlenecks
    """
    print(f"\n{'='*80}")
    print(f"PROFILING AGENT PIPELINE")
    print(f"{'='*80}")
    print(f"Question: {question}")
    print(f"Iterations: {iterations}")
    print(f"{'='*80}\n")
    
    timings = {
        'ambiguity_check': [],
        'sql_generation': [],
        'sql_execution': [],
        'visualization': [],
        'total': []
    }
    
    for i in range(iterations):
        print(f"Iteration {i+1}/{iterations}...")
        
        # Total time
        total_start = time.time()
        
        # Ambiguity check
        amb_start = time.time()
        ambiguity_checker.check_for_ambiguity(question)
        amb_time = time.time() - amb_start
        timings['ambiguity_check'].append(amb_time)
        
        # SQL generation
        sql_start = time.time()
        sql_result = sql_generator.generate_sql(question)
        sql_time = time.time() - sql_start
        timings['sql_generation'].append(sql_time)
        
        if sql_result.get('success'):
            # SQL execution
            exec_start = time.time()
            query_result = db_service.execute_query(sql_result['sql'])
            exec_time = time.time() - exec_start
            timings['sql_execution'].append(exec_time)
            
            # Visualization
            if query_result.get('success'):
                viz_start = time.time()
                viz_recommender.recommend_visualization(
                    sql=sql_result['sql'],
                    columns=query_result.get('columns', []),
                    results=query_result.get('rows', []),
                    row_count=query_result.get('row_count', 0)
                )
                viz_time = time.time() - viz_start
                timings['visualization'].append(viz_time)
        
        total_time = time.time() - total_start
        timings['total'].append(total_time)
        
        time.sleep(0.5)  # Rate limiting
    
    # Calculate averages
    print(f"\n📊 PERFORMANCE PROFILE:\n")
    for stage, times in timings.items():
        if times:
            avg = sum(times) / len(times)
            min_time = min(times)
            max_time = max(times)
            print(f"{stage:20s}: avg={avg:.3f}s  min={min_time:.3f}s  max={max_time:.3f}s")
    
    return timings

# Profile a query
# profile_agent_pipeline("How many patients do we have?", iterations=3)

## 9. Semantic Dictionary & Templates Inspection

View and analyze the semantic layer

In [ ]:
# View semantic dictionary
print("SEMANTIC DICTIONARY")
print("=" * 80)
semantic_df = pd.DataFrame(semantic_dict)
print(semantic_df.to_string(index=False))
print(f"\nTotal terms: {len(semantic_dict)}")

In [ ]:
# View query templates
print("\nQUERY TEMPLATES")
print("=" * 80)
for i, template in enumerate(query_templates, 1):
    print(f"\n{i}. {template.get('name', 'Unnamed')}")
    print(f"   Description: {template.get('description', 'N/A')}")
    print(f"   SQL Template:\n{template.get('sql_template', 'N/A')[:200]}...")
    print("-" * 80)

In [ ]:
# View database schemas
print("\nDATABASE VIEW SCHEMAS")
print("=" * 80)
for view_name, columns in view_schemas.items():
    print(f"\n{view_name} ({len(columns)} columns)")
    print(f"   Columns: {', '.join(columns[:10])}..." if len(columns) > 10 else f"   Columns: {', '.join(columns)}")
    print("-" * 80)

## 10. Agent Development Scratchpad

Use the cells below for ad-hoc testing and experimentation

In [ ]:
# Test your own queries here


In [ ]:
# Experiment with prompts


In [ ]:
# Test edge cases


In [ ]:
# Profile performance


In [ ]:
# Debug agent behavior
